# W1D2: 색공간 변환 + ROI (Region of Interest)

> 📖 원리 이해: [LearnOpenCV - Color Spaces in OpenCV](https://learnopencv.com/color-spaces-in-opencv-cpp-python/)
> 📋 파라미터 확인: [OpenCV 공식 - cvtColor](https://docs.opencv.org/4.x/d8/d01/group__imgproc__color__conversions.html)
> 📋 파라미터 확인: [OpenCV 공식 - resize](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231b7cb886d12)

## 📌 오늘의 목표
- BGR, Grayscale, HSV 색공간의 차이를 이해하고 변환하기
- ROI 슬라이싱으로 결함 영역만 잘라서 확대 분석
- resize로 이미지 크기 조절하기

## 🎯 배울 함수
| 함수 | 설명 |
|------|------|
| `cv2.cvtColor(img, code)` | 색공간 변환 (BGR→Gray, BGR→HSV 등) |
| `img[y1:y2, x1:x2]` | ROI 슬라이싱 — 관심 영역 잘라내기 |
| `cv2.resize(img, (w, h))` | 이미지 크기 변경 |

## 📦 import + 데이터 로딩

D1에서 사용한 동일한 설정 + 결함 이미지 1장을 미리 로딩합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path("../data/kaggle/archive/NEU-DET/train/images")
DEFECT_TYPES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

img_bgr = cv2.imread(str(DATA_DIR / "inclusion" / "inclusion_1.jpg"))
print(f"로딩 완료: shape={img_bgr.shape}, dtype={img_bgr.dtype}")

## 🔧 1. BGR → Grayscale 변환

**`cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)`** 로 흑백 변환합니다.
- 3채널(BGR) → 1채널(밝기만) 으로 줄어듦
- 공식: `Gray = 0.299*R + 0.587*G + 0.114*B` (사람 눈이 녹색에 민감하므로 G 가중치가 높음)

**제조 검사 연결:** AOI/SPI 검사에서 결함 검출은 대부분 Grayscale에서 시작합니다. 색상보다 **밝기 차이**가 결함 판별의 핵심이기 때문입니다.

In [ ]:
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print(f"BGR  shape: {img_bgr.shape}  → 3채널")
print(f"Gray shape: {img_gray.shape}    → 1채널")
print(f"메모리: BGR {img_bgr.nbytes:,} bytes → Gray {img_gray.nbytes:,} bytes (1/3)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("BGR (Original)", fontsize=14)
axes[0].axis('off')

axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title("Grayscale", fontsize=14)
axes[1].axis('off')

plt.suptitle("inclusion_1: BGR vs Grayscale", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- Grayscale에서도 inclusion(이물질) 결함이 충분히 보임 → 밝기 차이만으로 검출 가능한 결함
- `cmap='gray'` 를 안 넣으면 matplotlib이 컬러맵을 적용해서 이상한 색으로 나옴 (주의!)

> 🧪 **실험해보기:** `cmap='gray'`를 빼고 `axes[1].imshow(img_gray)`만 실행해보세요. 어떻게 보이나요?

## 🔧 2. BGR → HSV 변환

**`cv2.cvtColor(img, cv2.COLOR_BGR2HSV)`** 로 HSV 색공간으로 변환합니다.
- **H (Hue)**: 색상 (0~179). 빨강=0, 초록=60, 파랑=120
- **S (Saturation)**: 채도 (0~255). 0이면 무채색(회색), 255이면 순수한 색
- **V (Value)**: 명도 (0~255). 0이면 검정, 255이면 밝음

**제조 검사 연결:** HSV는 "조명 변화에 강한" 색공간입니다. 공장에서 조명이 바뀌어도 V(밝기)만 변하고 H(색상)는 유지되므로, 색상 기반 검사에서 유리합니다.

In [ ]:
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

h, s, v = cv2.split(img_hsv)

print(f"HSV shape: {img_hsv.shape}")
print(f"H 범위: {h.min()} ~ {h.max()} (색상)")
print(f"S 범위: {s.min()} ~ {s.max()} (채도)")
print(f"V 범위: {v.min()} ~ {v.max()} (명도)")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (BGR→RGB)", fontsize=13)

axes[1].imshow(h, cmap='hsv')
axes[1].set_title(f"H (Hue) 색상\nrange: {h.min()}~{h.max()}", fontsize=13)

axes[2].imshow(s, cmap='gray')
axes[2].set_title(f"S (Saturation) 채도\nrange: {s.min()}~{s.max()}", fontsize=13)

axes[3].imshow(v, cmap='gray')
axes[3].set_title(f"V (Value) 명도\nrange: {v.min()}~{v.max()}", fontsize=13)

for ax in axes:
    ax.axis('off')

plt.suptitle("inclusion_1: HSV 채널 분리", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **H 채널**: NEU 철강 이미지는 대부분 무채색(회색 금속)이라 H 값이 의미 없을 수 있음
- **S 채널**: 채도가 낮으면 회색 계열 → 철강 표면은 S가 전반적으로 낮을 것
- **V 채널**: Grayscale과 비슷하게 보임. 결함 부위의 밝기 차이가 여기서 드러남

> 🤔 **판단 질문:** NEU 철강 데이터에서 HSV 중 어떤 채널이 결함 검출에 가장 유용할까요? (힌트: 철강은 무채색)

## 🔧 3. 6종 결함 × 3색공간 비교

결함 유형별로 Gray와 V채널이 어떻게 다른지 한눈에 비교합니다.

**제조 검사 연결:** 어떤 색공간이 특정 결함을 가장 잘 드러내는지 파악하는 것이 알고리즘 설계의 첫걸음입니다.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(15, 28))
col_titles = ["Original (BGR→RGB)", "Grayscale", "HSV - V채널"]

for row, defect in enumerate(DEFECT_TYPES):
    path = DATA_DIR / defect / f"{defect}_1.jpg"
    img = cv2.imread(str(path))
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv_v = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 2]
    
    axes[row, 0].imshow(img_rgb)
    axes[row, 1].imshow(gray, cmap='gray')
    axes[row, 2].imshow(hsv_v, cmap='gray')
    
    axes[row, 0].set_ylabel(defect, fontsize=13, fontweight='bold', rotation=0, labelpad=100)
    
    for col in range(3):
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(col_titles[col], fontsize=14, fontweight='bold')

plt.suptitle("6종 결함 × 색공간 비교", fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 해석 가이드
- **Grayscale vs V채널**: 거의 비슷하게 보이지만 미세한 차이가 있음. 철강처럼 무채색 이미지에서는 Gray ≈ V
- **결함별 특징**: scratches는 선형, patches는 면적, crazing은 미세 패턴 — 색공간보다 **텍스처**가 핵심
- 결론: NEU 철강 데이터는 **Grayscale만으로도 충분**. HSV는 컬러 제품 검사(PCB, 식품 등)에서 더 빛남

> 🧪 **실험해보기:** S(채도) 채널도 3번째 열에 추가해서 비교해보세요. `hsv_v`를 `hsv_s`로 바꾸고 인덱스를 `[:, :, 1]`로!

## 🔧 4. ROI (Region of Interest) 슬라이싱

**`img[y1:y2, x1:x2]`** 로 이미지의 특정 영역만 잘라냅니다.
- numpy 배열이므로 슬라이싱 문법 그대로 사용
- **순서 주의**: `[행(y), 열(x)]` — height 먼저, width 나중!
- 잘라낸 영역은 독립적인 numpy 배열로 사용 가능

**제조 검사 연결:** AOI에서 전체 기판을 검사하지 않고, 관심 영역(패드, 부품, 납땜 부위)만 잘라서 집중 검사합니다. ROI 설정이 검사 속도와 정확도를 좌우합니다.

In [ ]:
img_scratch = cv2.imread(str(DATA_DIR / "scratches" / "scratches_1.jpg"))
img_rgb = cv2.cvtColor(img_scratch, cv2.COLOR_BGR2RGB)

h, w = img_scratch.shape[:2]
print(f"원본 크기: {w}(W) x {h}(H)")

roi = img_rgb[50:150, 80:180]
print(f"ROI 크기: {roi.shape[1]}(W) x {roi.shape[0]}(H)")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_rgb)
axes[0].set_title(f"Original ({w}x{h})", fontsize=13)
axes[0].axis('off')

img_with_rect = img_rgb.copy()
cv2.rectangle(img_with_rect, (80, 50), (180, 150), (255, 0, 0), 2)
axes[1].imshow(img_with_rect)
axes[1].set_title("ROI 위치 표시 (빨간 박스)", fontsize=13)
axes[1].axis('off')

axes[2].imshow(roi)
axes[2].set_title(f"ROI 확대 ({roi.shape[1]}x{roi.shape[0]})", fontsize=13)
axes[2].axis('off')

plt.suptitle("scratches_1: ROI 슬라이싱", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- ROI로 잘라내면 결함 부위를 **확대해서** 자세히 볼 수 있음
- `img[y1:y2, x1:x2]` 순서 헷갈리면: **"y가 먼저, x가 나중"** 으로 외우기
- `cv2.rectangle`의 좌표는 반대로 `(x, y)` 순서 — OpenCV 함수는 (x, y), numpy는 [y, x]!

> 🧪 **실험해보기:** ROI 좌표를 `[0:100, 0:100]` (좌상단), `[100:200, 100:200]` (중앙) 등으로 바꿔보세요. 결함이 있는 영역을 잘 잡을 수 있나요?

## 🔧 5. 4분할 ROI로 결함 위치 탐색

이미지를 4등분해서 결함이 어느 영역에 집중되어 있는지 파악합니다.

**제조 검사 연결:** 실제 AOI에서도 기판을 영역별로 나눠서 검사하고, 불량 위치의 통계를 냅니다. "어디에 결함이 많은가"는 공정 개선의 핵심 정보입니다.

In [ ]:
img = cv2.imread(str(DATA_DIR / "pitted_surface" / "pitted_surface_1.jpg"))
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

h, w = img_gray.shape
mid_h, mid_w = h // 2, w // 2

quadrants = {
    "좌상": img_gray[0:mid_h, 0:mid_w],
    "우상": img_gray[0:mid_h, mid_w:w],
    "좌하": img_gray[mid_h:h, 0:mid_w],
    "우하": img_gray[mid_h:h, mid_w:w],
}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

axes[0].imshow(img_rgb)
axes[0].set_title("Original", fontsize=13)
axes[0].axhline(y=mid_h, color='r', linewidth=1)
axes[0].axvline(x=mid_w, color='r', linewidth=1)
axes[0].axis('off')

for i, (name, quad) in enumerate(quadrants.items()):
    axes[i+1].imshow(quad, cmap='gray')
    mean_val = quad.mean()
    std_val = quad.std()
    axes[i+1].set_title(f"{name}\nmean={mean_val:.1f}, std={std_val:.1f}", fontsize=12)
    axes[i+1].axis('off')

plt.suptitle("pitted_surface_1: 4분할 ROI 분석", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("▶ std(표준편차)가 높은 영역 = 밝기 변화가 큰 영역 = 결함 가능성 높음")

### 해석 가이드
- **mean (평균)**: 영역의 전체 밝기. 높으면 밝은 영역, 낮으면 어두운 영역
- **std (표준편차)**: 밝기의 변동폭. **std가 높으면 텍스처가 복잡 → 결함 가능성**
- 실무에서는 이런 통계값을 기준으로 "이 영역을 더 자세히 검사할지" 결정함

> 🤔 **판단 질문:** 4개 영역 중 std가 가장 높은 곳이 실제로 결함이 집중된 곳인가요? 원본과 비교해보세요.

## 🔧 6. resize — 이미지 크기 변경

**`cv2.resize(img, (width, height), interpolation)`** 으로 크기를 변경합니다.
- `(width, height)` 순서 — shape과 반대! **(W, H) 순서 주의**
- **보간법(interpolation)**: 픽셀을 늘리거나 줄일 때 값을 계산하는 방법
  - `INTER_NEAREST`: 가장 가까운 픽셀 복사 (빠르지만 계단현상)
  - `INTER_LINEAR`: 주변 4개 픽셀의 가중평균 (기본값, 일반적으로 충분)
  - `INTER_CUBIC`: 주변 16개 픽셀로 계산 (확대 시 부드러움)
  - `INTER_AREA`: 영역 기반 (축소 시 가장 좋음)

**제조 검사 연결:** 카메라 해상도와 검사 알고리즘의 입력 크기가 다를 때 resize가 필수입니다. 딥러닝 모델은 보통 224x224, 256x256 같은 고정 크기를 요구합니다.

In [ ]:
img = cv2.imread(str(DATA_DIR / "crazing" / "crazing_1.jpg"), cv2.IMREAD_GRAYSCALE)
print(f"원본: {img.shape}")

img_half = cv2.resize(img, (img.shape[1] // 2, img.shape[0] // 2))
img_double = cv2.resize(img, (img.shape[1] * 2, img.shape[0] * 2))
img_square = cv2.resize(img, (224, 224))

print(f"절반 축소: {img_half.shape}")
print(f"2배 확대:  {img_double.shape}")
print(f"224x224:   {img_square.shape}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_half, cmap='gray')
axes[0].set_title(f"절반 축소\n{img_half.shape[1]}x{img_half.shape[0]}", fontsize=13)

axes[1].imshow(img, cmap='gray')
axes[1].set_title(f"원본\n{img.shape[1]}x{img.shape[0]}", fontsize=13)

axes[2].imshow(img_square, cmap='gray')
axes[2].set_title(f"224x224 (딥러닝 입력용)\n정사각형", fontsize=13)

for ax in axes:
    ax.axis('off')

plt.suptitle("crazing_1: resize 비교", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- 축소하면 미세한 결함(crazing 같은)이 사라질 수 있음 → 검사 정밀도 하락
- 224x224로 정사각형 변환 시 원본 비율이 달라짐 → 비율 유지가 필요한 경우 주의
- **축소 = 정보 손실**, **확대 = 정보 추가 없이 보간** — 둘 다 원본보다 나을 수 없음

> 🧪 **실험해보기:** `crazing`을 `inclusion`으로 바꿔보세요. 축소해도 결함이 잘 보이는 유형은?

## 🔬 파라미터 실험: 보간법 비교

ROI를 작게 잘라서 크게 확대할 때, 보간법에 따라 결과가 어떻게 달라지는지 비교합니다.

In [ ]:
img = cv2.imread(str(DATA_DIR / "scratches" / "scratches_1.jpg"), cv2.IMREAD_GRAYSCALE)

roi_small = img[80:110, 100:130]

interpolations = {
    "NEAREST": cv2.INTER_NEAREST,
    "LINEAR": cv2.INTER_LINEAR,
    "CUBIC": cv2.INTER_CUBIC,
    "AREA": cv2.INTER_AREA,
}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

axes[0].imshow(roi_small, cmap='gray')
axes[0].set_title(f"ROI 원본\n{roi_small.shape[1]}x{roi_small.shape[0]}px", fontsize=12)
axes[0].axis('off')

for i, (name, interp) in enumerate(interpolations.items()):
    roi_big = cv2.resize(roi_small, (150, 150), interpolation=interp)
    axes[i+1].imshow(roi_big, cmap='gray')
    axes[i+1].set_title(f"{name}\n150x150px", fontsize=12)
    axes[i+1].axis('off')

plt.suptitle("30x30 ROI → 150x150 확대: 보간법 비교", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **NEAREST**: 블록 느낌 (계단현상). 빠르지만 거칠음. 마스크/라벨 이미지에 적합
- **LINEAR**: 부드러운 결과. 일반적인 기본값으로 충분
- **CUBIC**: LINEAR보다 약간 더 부드러움. 확대 시 추천
- **AREA**: 축소에 최적화. 확대에 쓰면 NEAREST와 비슷

> **실무 요약**: 확대 → CUBIC, 축소 → AREA, 빠르게 → LINEAR, 마스크/라벨 → NEAREST

## 📝 정리

### 오늘 배운 것
| 함수 | 핵심 포인트 |
|------|------------|
| `cv2.cvtColor()` | BGR→Gray, BGR→HSV 변환. 색공간마다 용도가 다름 |
| `cv2.split()` | 다채널 이미지를 채널별로 분리 (H, S, V 각각 보기) |
| `img[y1:y2, x1:x2]` | ROI 슬라이싱. **y가 먼저, x가 나중** (numpy 규칙) |
| `cv2.resize()` | 크기 변경. **(W, H) 순서** — shape과 반대! |
| 보간법 | 확대=CUBIC, 축소=AREA, 기본=LINEAR, 마스크=NEAREST |

### 핵심 개념
- **Grayscale**: 결함 검출의 기본. 밝기 차이로 결함을 찾는 첫 단계
- **HSV**: 조명 변화에 강함. 컬러 제품 검사에서 유리 (철강에는 Gray로 충분)
- **ROI**: 관심 영역만 잘라서 집중 분석 → 검사 속도↑ 정확도↑
- **좌표 순서 함정**: numpy는 `[y, x]`, OpenCV 함수는 `(x, y)`, resize는 `(W, H)`

### 복습 퀴즈
1. `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)`의 결과 shape은 원본 대비 어떻게 달라지는가?
2. HSV에서 H, S, V는 각각 무엇의 약자이며, 제조 검사에서 V채널이 중요한 이유는?
3. `img[50:150, 80:180]`에서 50:150은 ___ 방향, 80:180은 ___ 방향이다. (가로/세로)
4. `cv2.resize(img, (224, 224))`에서 (224, 224)는 (__, __) 순서이다. (W,H / H,W)

### 다음에 할 것 (W1D3)
- 히스토그램 — `calcHist`로 밝기 분포를 그래프로 시각화
- 결함 유형별 히스토그램 패턴 비교